**API** — это специальные разделы сайта, где информацию можно получать без разметки, а формат запросов и ответов зафиксирован. API созданы для того, чтобы облегчить взаимодействие с сайтом для сторонних разработчиков.

Для того чтобы начать работать с API, обычно необходимо получить сервисный ключ авторизации — **токен**.

**Токен** — это средство идентификации пользователя или отдельного сеанса работы в компьютерных сетях и приложениях. Различают *программные* и *аппаратные* токены.

Мы будем использовать *программный* токен, который обычно представляет собой зашифрованную последовательность символов, позволяющую точно идентифицировать объект и определить уровень его привилегий. Он генерируется системой авторизации и привязывается к конкретному сеансу работы, клиенту сети или пакету данных.

### Запрос к API из кода

In [1]:
import requests # Импортируем модуль requests
token = '8a485b048a485b048a485b04bc8977339e88a488a485b04e3f070135a85d173fb658c39' # Указываем свой сервисный токен
url = 'https://api.vk.com/method/users.get' # Указываем адрес страницы к которой делаем запрос
params = {'user_id': 1, 'v': 5.95, 'fields': 'sex,bdate', 'access_token': token, 'lang': 'ru'} # Перечисляем параметры нашего запроса в словаре params
response = requests.get(url, params=params) # Отправляем запрос
print(response.text) # Выводим текст ответа на экран

{"response":[{"id":1,"bdate":"10.10.1984","sex":2,"first_name":"Павел","last_name":"Дуров","can_access_closed":true,"is_closed":false}]}


Мы получили строку в ```JSON-формате```, которую можно преобразовать в словарь с помощью метода ```json()```, после чего можно с лёгкостью обращаться к различным полям.

Словари нагляднее выводить с помощью функции ```pprint()```, которую мы уже использовали ранее:

In [2]:
from pprint import pprint # Импортируем функцию pprint()
pprint(response.json()) # Выводим содержимое словаря, содержащего ответ, на экран

{'response': [{'bdate': '10.10.1984',
               'can_access_closed': True,
               'first_name': 'Павел',
               'id': 1,
               'is_closed': False,
               'last_name': 'Дуров',
               'sex': 2}]}


Как вы видите, по ключу response мы можем получить список, в котором хранятся словари, содержащие информацию о запрошенных нами пользователях. Мы запросили информацию лишь об одном из них, поэтому список содержит только один элемент. Извлечём его:

In [3]:
user = response.json()['response'][0] # Извлекаем из словаря по ключу response информацию о первом пользователе
print(user['bdate']) # Выводим дату рождения первого пользователя на экран

10.10.1984


>Задание 6.1
1 point possible (graded)
Мы вывели на экран дату рождения пользователя. Напишите строку кода, которая выведет на экран имя пользователя:

In [4]:
print(user['first_name'])

Павел


Метод ```users.get()``` позволяет запрашивать информацию о множестве (до 1 000) пользователей одновременно. Для этого нужно использовать параметр ```user_ids``` и передавать id через запятую в строковом формате. Например, чтобы получить информацию о пользователях с id=1, id=2, id=3, необходимо передать значение параметра user_ids='1,2,3'.

Попробуем это сделать:

In [6]:
ids = ",".join(map(str, range(1, 4))) # Формируем строку, содержащую информацию о поле id первых трёх пользователей
params = {'user_ids': ids, 'v': 5.95, 'fields': 'bday', 'access_token': token, 'lang': 'ru'} # Формируем строку параметров
pprint(requests.get(url, params=params).json()) # Посылаем запрос, полученный ответ в формате JSON-строки преобразуем в словарь 
# и выводим на экран его содержимое, используя функцию pprint()

{'response': [{'can_access_closed': True,
               'first_name': 'Павел',
               'id': 1,
               'is_closed': False,
               'last_name': 'Дуров'},
              {'can_access_closed': False,
               'first_name': 'Александра',
               'id': 2,
               'is_closed': True,
               'last_name': 'Владимирова'},
              {'can_access_closed': True,
               'deactivated': 'deleted',
               'first_name': 'DELETED',
               'id': 3,
               'is_closed': False,
               'last_name': ''}]}


>Задание 6.2
 1 point possible (graded)
 Используя API, определите долю женщин (sex=1) среди пользователей с id от 1 до 500. Иногда будут попадаться пользователи, у которых пол не указан (sex=0), — таких пользователей не нужно учитывать в общем числе.
 В ответе укажите число, округлив до двух знаков после точки-разделителя, например, 0.55.
 Пример: если у нас будет 300 пользователей с sex=1, 100 пользователей с sex=2 и 100 пользователей с sex=0, то в ответе должно быть 0.75.

In [ ]:
import requests
#token = '...'
url = 'https://api.vk.com/method/users.get'
ids = ",".join(map(str, range(1, 501)))
params = {'user_ids': ids, 'v': 5.95, 'fields': 'sex,bdate', 'access_token': token, 'lang': 'ru'}
response = requests.get(url, params=params).json()['response']

men=women=0
for elem in response:
    if elem['sex'] == 2:
        men+=1
    elif elem['sex'] == 1:
        women+=1
    else:
        continue
print(round(women/(men+women),2))

0.49


### Сбор информации из групп

Стоит отметить, что есть много сервисов, которые выгружают похожую статистику из соцсетей. Однако им свойственны недостатки универсальных решений:
- не учитываются все особенности вашего проекта;
- используется фиксированный набор метрик, дополнительную обработку данных приходится делать вам;
- не всегда бесплатны и вряд ли позволят работать с большими объёмами данных.

Теперь мы научимся считать произвольные метрики групп, собирая данные из API и работая с двумя ограничениями, которые свойственны практически всем системам:
- ограничение на количество вызовов в единицу времени;
- ограничение на количество выгружаемых строк за один запрос.

Давайте рассмотрим, как работать с этими ограничениями на примере выгрузки списка пользователей [группы](https://vk.com/vk) социальной сети ВКонтакте.

Обратимся к [документации](https://dev.vk.com/ru/?ref=old_portal#products-and-technologies), чтобы узнать, какие методы нам доступны для групп, — для получения списка пользователей группы доступен метод [groups.getMembers](https://dev.vk.com/ru/method/groups.getMembers?ref=old_portal).

Согласно документации, обязательным параметром данного метода является group_id — идентификатор, или короткое имя, группы. В нашем случае это vk: https://vk.com/vk. Протестируем, как работает метод в самом простом случае, — получим id участников группы:

In [8]:
import requests # Импортируем модуль requests
#token = ' ... ' # Указываем свой сервисный токен
url = 'https://api.vk.com/method/groups.getMembers' # Указываем адрес обращения
params = {'group_id': 'vk', 'v': 5.95, 'access_token': token} # Формируем строку параметров
response = requests.get(url, params = params) # Посылаем запрос
data = response.json() # Ответ сохраняем в переменной data в формате словаря
print(data) # Выводим содержимое переменной data на экран (отображён фрагмент)

{'response': {'count': 20487035, 'items': [200000000175, 200000000200, 200000000212, 200000000214, 200000000230, 200000000252, 200000005322, 200000510128, 200000510190, 200000510246, 5, 6, 7, 14, 19, 34, 46, 47, 54, 68, 79, 88, 102, 106, 161, 163, 166, 177, 198, 212, 219, 243, 259, 271, 279, 296, 302, 316, 319, 334, 344, 345, 353, 354, 385, 403, 404, 421, 431, 433, 450, 467, 485, 510, 513, 550, 619, 628, 639, 640, 648, 660, 670, 690, 696, 702, 721, 741, 744, 804, 809, 832, 834, 847, 900, 905, 907, 912, 914, 921, 943, 952, 958, 966, 976, 997, 1000, 1008, 1011, 1018, 1023, 1033, 1039, 1045, 1058, 1059, 1063, 1091, 1097, 1115, 1127, 1128, 1131, 1139, 1140, 1149, 1159, 1164, 1174, 1179, 1181, 1185, 1188, 1207, 1213, 1245, 1270, 1273, 1301, 1322, 1333, 1334, 1351, 1381, 1386, 1388, 1406, 1411, 1417, 1418, 1432, 1470, 1490, 1498, 1503, 1529, 1531, 1550, 1568, 1570, 1575, 1586, 1590, 1593, 1610, 1615, 1632, 1634, 1635, 1650, 1665, 1674, 1679, 1690, 1697, 1698, 1699, 1700, 1721, 1725, 1740, 17

По ключу count мы можем получить общее число участников группы, а список по ключу items хранит их id. Посмотрим на него поближе:

In [9]:
print(len(data['response']['items'])) # Выводим на экран количество элементов словаря

1000


Мы видим, что всего пользователей в группе больше 11 миллионов, а получили мы только первую тысячу пользователей группы. По информации, указанной в документации о параметре count, это максимум, который может отдать API за один раз.

Для получения следующей тысячи пользователей можно воспользоваться параметром offset (с англ. смещение), который передвинет начало отсчёта. Для выгрузки всех пользователей группы будем в цикле выгружать по 1000 пользователей (count будет всегда равен 1000), увеличивая смещение offset на величину count.

Для тренировки напишем цикл выгрузки первых 20 пользователей со значением count=5. Иными словами, мы будем выгружать по пять пользователей за запрос до тех пор, пока не получим информацию о 20 пользователях.

Давайте выведем на экран первые 20 пользователей из нашей первой попытки получить информацию о 1000 пользователей, чтобы мы могли сверить результат выгрузки из 20 пользователей:

In [10]:
users_for_checking = data['response']['items'][:20] # Загружаем в переменную информацию об id первых 20 пользователей в виде списка
print(users_for_checking) # Выводим перечень id первых 20 пользователей

[200000000175, 200000000200, 200000000212, 200000000214, 200000000230, 200000000252, 200000005322, 200000510128, 200000510190, 200000510246, 5, 6, 7, 14, 19, 34, 46, 47, 54, 68]


Теперь используем count и offset, чтобы получить те же id по пять за раз:

In [11]:
import requests # Импортируем модуль requests
#token = ' ... ' # Указываем свой сервисный токен
url = 'https://api.vk.com/method/groups.getMembers' # Указываем адрес обращения
count = 5 
offset = 0 
user_ids = [] 
max_count = 20 
while offset < max_count: 
    # Будем выгружать по count=5 пользователей, 
    # начиная с того места, где закончили на предыдущей итерации (offset) 
    print('Выгружаю {} пользователей с offset = {}'.format(count, offset))   
    params = {'group_id': 'vk', 'v': 5.95, 'count': count, 'offset': offset, 'access_token': token} 
    response = requests.get(url, params = params) 
    data = response.json() 
    user_ids += data['response']['items'] 
    # Увеличиваем смещение на количество строк, которое мы уже выгрузили 
    offset += count 
print(user_ids) 

Выгружаю 5 пользователей с offset = 0
Выгружаю 5 пользователей с offset = 5
Выгружаю 5 пользователей с offset = 10
Выгружаю 5 пользователей с offset = 15
[200000000175, 200000000200, 200000000212, 200000000214, 200000000230, 200000000252, 200000005322, 200000510128, 200000510190, 200000510246, 5, 6, 7, 14, 19, 34, 46, 47, 54, 68]


Сравним списки, полученные двумя способами:

In [12]:
print(user_ids == users_for_checking) 

True


Так как результат сравнения — True, списки идентичны. Значит, второй способ работает корректно. Теперь мы можем получить данные обо всех пользователях, выставив count = 1000 и max_count = data['response']['count'].

### Ограничение по частоте запросов

В API часто добавляют ограничение по частоте запросов, чтобы отдельно взятые пользователи слишком сильно не перегружали сервер. Подобное ограничение есть и у ВКонтакте — в документации указано, что можно делать не более трёх запросов в секунду.

Чтобы не следить за частотой отправки запросов с секундомером в руках, мы можем после каждого запроса делать паузу. В этом случае, даже если код будет выполняться на самом быстром компьютере, мы не нарушим установленное ограничение, так как периодичность отправки запросов будет искусственно замедлена.

Воспользуемся библиотекой ```time``` и методом ```sleep```, с помощью которого мы можем добавить паузу, например в 0.5 секунд, после каждого запроса:

In [13]:
import requests # Импортируем модуль requests
import time # Импортируем модуль time
#token = ' ... ' # Указываем свой сервисный токен
url = 'https://api.vk.com/method/groups.getMembers' # Указываем адрес страницы, к которой делаем запрос
count = 1000 
offset = 0  
user_ids = []  
while offset < 5000: 
    params = {'group_id': 'vk', 'v': 5.95, 'count': count, 'offset': offset, 'access_token': token} 
    response = requests.get(url, params = params) 
    data = response.json() 
    user_ids += data['response']['items'] 
    offset += count 
    print('Ожидаю 0.5 секунды...') 
    time.sleep(0.5) 
print('Цикл завершен, offset =',offset) 

Ожидаю 0.5 секунды...
Ожидаю 0.5 секунды...
Ожидаю 0.5 секунды...
Ожидаю 0.5 секунды...
Ожидаю 0.5 секунды...
Цикл завершен, offset = 5000


## Лайки, репосты и комментарии

> Через API новостной ленты ВКонтакте мы можем получить информацию о взаимодействии с сообщениями в ленте.

Для примера продолжим работать с группой https://vk.com/vk и рассмотрим последние 100 сообщений в новостной ленте.

Примечание: обратите внимание, что т.к. сообщения в новостной ленте непрерывно обновляются, то ваш результат выполнения кода ниже будет отличаться от нашего варианта.

Для получения информации о сообщениях на стене в API ВКонтакте предусмотрен метод [wall.get](https://dev.vk.com/ru/method/wall.get?ref=old_portal). Применим его:

In [14]:
import requests # Импортируем модуль requests
from pprint import pprint # Импортируем функцию pprint()
#token = ' ... ' # Указываем свой сервисный токен
url = 'https://api.vk.com/method/wall.get' # Указываем адрес страницы, к которой делаем запрос
params = {'domain': 'vk', 'filter': 'owner', 'count': 1000, 'offset': 0, 'access_token': token, 'v': 5.95} 
response = requests.get(url, params = params) 
pprint(response.json()) 

{'response': {'count': 983,
              'items': [{'attachments': [{'photo': {'access_key': '0c1e22d5cff7dedaf0',
                                                    'album_id': -7,
                                                    'date': 1773667444,
                                                    'id': 457371002,
                                                    'orig_photo': {'height': 1920,
                                                                   'type': 'base',
                                                                   'url': 'https://sun9-49.userapi.com/s/v1/ig2/3AoP2y28oOMT2S152U04r-x6wtgvq39Mx625KwLuAnwbDXHKZ15CuGCrhk-fam4-V8yb_WABVdnkaLfqlRFxtiU8.jpg?quality=95&crop=0,0,1920,1920&as=32x32,48x48,72x72,108x108,160x160,240x240,360x360,480x480,540x540,640x640,720x720,1080x1080,1280x1280,1440x1440,1920x1920&from=bu',
                                                                   'width': 1920},
                                                    'own

Посмотрим на количество результатов:

In [15]:
len(response.json()['response']['items'])

100

Посмотрим на информацию об отдельном сообщении:

In [16]:
response.json()['response']['items'][0]

{'inner_type': 'wall_wallpost',
 'comments': {'count': 39},
 'marked_as_ads': 0,
 'hash': '0EHgiJglXTpu3h29Tg',
 'type': 'post',
 'push_subscription': {'is_subscribed': False},
 'attachments': [{'type': 'photo',
   'photo': {'album_id': -7,
    'date': 1773667444,
    'id': 457371002,
    'owner_id': -22822305,
    'access_key': '0c1e22d5cff7dedaf0',
    'post_id': 1668201,
    'sizes': [{'height': 72,
      'type': 's',
      'width': 72,
      'url': 'https://sun9-49.userapi.com/s/v1/ig2/3AoP2y28oOMT2S152U04r-x6wtgvq39Mx625KwLuAnwbDXHKZ15CuGCrhk-fam4-V8yb_WABVdnkaLfqlRFxtiU8.jpg?quality=95&crop=0,0,1920,1920&as=32x32,48x48,72x72,108x108,160x160,240x240,360x360,480x480,540x540,640x640,720x720,1080x1080,1280x1280,1440x1440,1920x1920&from=bu&cs=72x0'},
     {'height': 160,
      'type': 'm',
      'width': 160,
      'url': 'https://sun9-49.userapi.com/s/v1/ig2/3AoP2y28oOMT2S152U04r-x6wtgvq39Mx625KwLuAnwbDXHKZ15CuGCrhk-fam4-V8yb_WABVdnkaLfqlRFxtiU8.jpg?quality=95&crop=0,0,1920,1920&as=3

В полях comments, likes и reposts содержится статистика по взаимодействию с сообщением пользователей (на момент получения информации) — число комментариев, лайков и репостов.

Давайте соберём итоговую статистику для последних десяти непустых сообщений в словарь stats. В качестве ключа будем использовать начало сообщения (если начало сообщения пустое, то информацию о таком сообщении проигнорируем), в качестве значения — список с тремя интересующими нас метриками и временем публикации (комментарии, лайки, репосты, дата публикации):

In [17]:
stats = {} 
count_post = 0 # Счётчик «непустых» сообщений
for record in response.json()['response']['items'][:]:
    title = record['text'][:30] 
    if title: 
        stats[title] = [record['comments']['count'], record['likes']['count'], record['reposts']['count'], record['date']] 
        count_post += 1 
    if count_post < 10: 
        continue 
    else: 
        break 
pprint(stats)

{'8 Марта уже завтра \U0001fa77\n\nВаших бл': [172, 481, 62, 1772878800],
 '«Форум в большом городе» — в п': [113, 302, 66, 1772692200],
 'В VK Шагах — последний и очень': [46, 219, 17, 1772868600],
 'Вот и результаты [https://vk.c': [79, 216, 11, 1772645414],
 'Если партнёр досмотрит сериал ': [39, 104, 24, 1773667444],
 'Какой стикер чаще всего отправ': [427, 243, 16, 1773559560],
 'Мм, ага, вот и новый розыгрыш ': [339, 278, 58, 1773308726],
 'План на март: классно проводит': [416, 440, 26, 1772953201],
 'Попробуем договориться с погод': [1692, 308, 31, 1772808642],
 'Сила, вдохновение и энергия — ': [191, 280, 38, 1772727300]}


ДОПОЛНИТЕЛЬНО

Если вы размещаете рекламу во ВКонтакте, то можно выгружать всю статистику через [ads API](https://dev.vk.com/ru/?ref=old_portal).

Полный список методов ВКонтакте можно посмотреть в [документации](https://dev.vk.com/ru/method?ref=old_portal).

### Другие API

Вы познакомились с интерфейсами прикладного программирования — API (на примере API социальной сети ВКонтакте).

API для разработчиков предоставляют и многие другие платформы. Вот список, пожалуй, самых популярных из них:

* Google Maps API
* YouTube API
* Twitter API
* Facebook API

Вы также можете воспользоваться интернет-поиском, указав в поисковой строке, например, «курсы валют API» или «прогноз погоды api», — среди первых результатов выдачи чаще всего с лёгкостью можно найти ссылки на необходимый функционал.